## 4 Финальная предобработка

**Входы**

- Train, validation и test после фиксированной очистки и разбиения.
- Классификация 7 числовых, 3 номинальных/бинарных и 1 порядкового признака.

**Выходы**

- X_train / y_train и X_validation / y_validation.
- Шаблон ColumnTransformer для изолированного обучения внутри каждого Pipeline.

**Что делаем**

Фиксируем состав и правила преобразований. Статистики оцениваем только на train; test до этапа 6 не преобразуем.

In [ ]:
# Только для разработки: открыть раздел можно из notebooks/ или корня проекта
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


| Группа | Преобразования | Обоснование |
| --- | --- | --- |
| Числовые | Медиана train, индикаторы пропусков, RobustScaler по медиане и межквартильному размаху train | Правые хвосты и разные масштабы; не удаляем редкие высокие доходы |
| Номинальные, включая бинарный прошлый дефолт | Мода train, one-hot; неизвестные категории — нулевой вектор соответствующего блока | Не задаём искусственный порядок |
| Порядковый loan_grade | Мода train, A=0 … G=6 | Сохраняем естественный порядок |

Для логистической регрессии код грейда означает дополнительное предположение об одинаковом шаге в логарифме шансов; это ограничение линейного baseline. Деревья используют пороговые разбиения по порядку, не требуя линейной формы эффекта. Масштабирование не меняет порядок значений для деревьев.

Индикаторы пропусков создаются для столбцов с пропусками на train. Заполнять ставку из loan_grade без отдельного обоснования не будем. Долю кредита не пересчитываем: обнаружены расхождения с формулой, их источник неизвестен. Целевая переменная исключается **до** построения преобразований.

In [ ]:
exec(compile(__import__('pathlib').Path('src/notebooks/_code/preprocessing.py').read_text(encoding='utf-8'), 'src/notebooks/_code/preprocessing.py', 'exec'), globals())

In [ ]:
from sklearn.base import clone
from threadpoolctl import threadpool_limits

# Ограничиваем число вычислительных потоков для воспроизводимых запусков.
_ml_thread_limit = threadpool_limits(limits=1)
assert set(FEATURES) == set(train.columns) - {'loan_status'}
assert set(NUMERICAL_FEATURES) == set(numerical_columns)
assert set(NOMINAL_FEATURES) == set(nominal_columns) - {'loan_status'}
assert set(ORDINAL_FEATURES) == set(ordinal_columns)
assert train.index.intersection(validation.index).empty
assert train.index.intersection(test.index).empty
assert validation.index.intersection(test.index).empty

X_train = train.loc[:, FEATURES].copy()
y_train = train['loan_status'].astype(int).copy()
X_validation = validation.loc[:, FEATURES].copy()
y_validation = validation['loan_status'].astype(int).copy()
preprocessor_template = make_preprocessor()
audit_preprocessor = clone(preprocessor_template).fit(X_train)
preprocessing_preview = pd.DataFrame(
    audit_preprocessor.transform(X_train.head(5)),
    columns=audit_preprocessor.get_feature_names_out(), index=X_train.head(5).index,
)
split_summary = pd.DataFrame([
    {'Выборка': name, 'Записей': len(frame), 'Доля дефолтов': frame['loan_status'].mean()}
    for name, frame in [('train', train), ('validation', validation), ('test', test)]
]).set_index('Выборка')
display(split_summary.round(4))
display(preprocessing_preview.round(3))